In [1]:
import pandas as pd
import random
import numpy as np
import h5py
import os
import pickle

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [2]:
test_ratio = 0.2
valid_ratio = 0.2
seq_len = 100

artificial_missing_rate = 0.2

path_daqing_all_well = 'RawData/NewZealand/logs.csv'
dataset_saving_dir = "../generated_datasets/NewZealand_seqlen100_02masked/"

In [3]:
df = pd.read_csv(path_daqing_all_well,sep=',')

df_well = df.groupby('WELLNAME').agg(['count'])
well_names = list(df_well.index)

test_well_names = random.sample(well_names, k=int(len(well_names) * test_ratio))
valid_well_names = random.sample(set(well_names) - set(test_well_names), k=int(len(well_names) * valid_ratio))
train_well_names = list(set(well_names) - set(test_well_names) - set(valid_well_names))

print('训练井：', train_well_names,len(train_well_names), '验证井：', valid_well_names, len(valid_well_names), '测试井：', test_well_names, len(test_well_names))

C:\Users\lumisong\AppData\Local\Temp\ipykernel_16096\2801081400.py:1: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_daqing_all_well,sep=',')


训练井： ['Cheal-A4', 'Kaimiro-1', 'Kapuni-6', 'Karewa-1', 'Totara-1', 'MB-V(3)', 'Karaka-A1', 'Toetoe-3', 'MR2P1-LL', 'Maari-2', 'Ahuroa-2A', 'Toetoe-5', 'Wharehuia-1', 'Tuhua-6', 'Tuhua-8A', 'Miromiro-1', 'Waihi-1A', 'Cheal-A3X', 'Ngaere-3', 'Tuhua-3C', 'Turangi-3', 'Pohokura-3', 'Waihapa-6A', 'Waihapa-3', 'Ahuroa-1A', 'Kapuni-8', 'Tui SW-2', 'Kahu-1', 'Ngatoro-6', 'Mangahewa-3', 'Toetoe-5A', 'Toru-1', 'Kaimiro-2', 'Surville-1', 'Waihapa-8', 'Kauri-F5', 'Republic New Plymouth-4', 'Pohokura South-1', 'Goss-A1', 'Tui-3H', 'Waihapa-1', 'MB-W(2)', 'Tahi-1', 'Toetoe-4A', 'Fresne-1', 'Kahili-1B', 'Momoho-1', 'Arawa-1', 'Kapuni-14', 'Tuhua-4', 'Kopuwai-1', 'Kaimiro-7ST1', 'Durham-1', 'MA-14A', 'Cardiff-1', 'Ngatoro-3ST1', 'Ngaere-2', 'Cheal-A6 ST1', 'Tariki-2C', 'MA-10A', 'Te Popo-1', 'Kapuni-7', 'Tui-2H', 'Cheal-A7', 'Hector-1', 'McKee-3', 'POB-02A', 'Maari-1', 'Maui-2', 'Ohanga-1', 'Tariki-2B', 'Tawa B-1 ST2', 'POW-02', 'Kapuni-11', 'Te Whatu-2', 'Toetoe-6B', 'Terrace-1', 'MB-R(1)', 'Ahuroa-4

In [4]:
# 打印划分井的数量
print('训练井：', len(train_well_names), '验证井：', len(valid_well_names), '测试井：', len(test_well_names))

训练井： 245 验证井： 81 测试井： 81


In [5]:
print("原始缺失率：", df.isnull().sum().sum() / (df.shape[0] * df.shape[1]))

原始缺失率： 0.2924751034488007


In [6]:
df.drop(columns=['ONSHORE', 'DIRSURVEY', 'FORMATION','X','Y','Z'], inplace=True)
df.rename(columns={'WELLNAME': 'WELL'}, inplace=True)

In [7]:
train_set_X = scaler.fit_transform(df[df['WELL'].isin(train_well_names)].drop(['WELL'], axis=1))
print("训练集标准化数据形状：", train_set_X.shape)
val_set_X = scaler.transform(df[df['WELL'].isin(valid_well_names)].drop(['WELL'], axis=1))
print("验证集标准化数据形状：", val_set_X.shape)
test_set_X = scaler.transform(df[df['WELL'].isin(test_well_names)].drop(['WELL'], axis=1))
print("测试集标准化数据形状：", test_set_X.shape)

训练集标准化数据形状： (3883807, 15)
验证集标准化数据形状： (1303791, 15)
测试集标准化数据形状： (1239780, 15)


In [8]:
def window_truncate(feature_vectors, seq_len, sliding_len=None):
    """ Generate time series samples, truncating windows from time-series data with a given sequence length.
    Parameters
    ----------
    feature_vectors: time series data, len(shape)=2, [total_length, feature_num]
    seq_len: sequence length
    sliding_len: size of the sliding window
    """
    sliding_len = seq_len if sliding_len is None else sliding_len
    total_len = feature_vectors.shape[0]
    start_indices = np.asarray(range(total_len // sliding_len)) * sliding_len
    if total_len - start_indices[-1] * sliding_len < seq_len:  # remove the last one if left length is not enough
        start_indices = start_indices[:-1]
    sample_collector = []
    for idx in start_indices:
        sample_collector.append(feature_vectors[idx: idx + seq_len])
    return np.asarray(sample_collector).astype('float32')

train_set_X = window_truncate(train_set_X, seq_len=seq_len)
val_set_X = window_truncate(val_set_X, seq_len=seq_len)
test_set_X = window_truncate(test_set_X, seq_len=seq_len)

In [9]:
def random_mask(vector, artificial_missing_rate):
    """generate indices for random mask"""
    assert len(vector.shape) == 1
    indices = np.where(~np.isnan(vector))[0].tolist()
    indices = np.random.choice(indices, int(len(indices) * artificial_missing_rate))
    return indices

def add_artificial_mask(X, artificial_missing_rate, set_name):
    """Add artificial missing values.
    Parameters
    ----------
    X: feature vectors
    artificial_missing_rate: rate of artificial missing values that are going to be create
    set_name: dataset name, train/val/test
    """
    sample_num, seq_len, feature_num = X.shape
    if set_name == "train":
        # if this is train set, we don't need add artificial missing values right now.
        # If we want to apply MIT during training, dataloader will randomly mask out some values to generate X_hat

        # calculate empirical mean for model GRU-D, refer to paper
        mask = (~np.isnan(X)).astype(np.float32)
        X_filledWith0 = np.nan_to_num(X)
        empirical_mean_for_GRUD = np.sum(mask * X_filledWith0, axis=(0, 1)) / np.sum(
            mask, axis=(0, 1)
        )
        data_dict = {
            "X": X,
            "empirical_mean_for_GRUD": empirical_mean_for_GRUD,
        }
    else:
        # if this is val/test set, then we need to add artificial missing values right now,
        # because we need they are fixed
        X = X.reshape(-1)
        indices_for_holdout = random_mask(X, artificial_missing_rate)
        X_hat = np.copy(X)
        X_hat[indices_for_holdout] = np.nan  # X_hat contains artificial missing values
        missing_mask = (~np.isnan(X_hat)).astype(np.float32)
        # indicating_mask contains masks indicating artificial missing values
        indicating_mask = ((~np.isnan(X_hat)) ^ (~np.isnan(X))).astype(np.float32)

        data_dict = {
            "X": X.reshape([sample_num, seq_len, feature_num]),
            "X_hat": X_hat.reshape([sample_num, seq_len, feature_num]),
            "missing_mask": missing_mask.reshape([sample_num, seq_len, feature_num]),
            "indicating_mask": indicating_mask.reshape(
                [sample_num, seq_len, feature_num]
            ),
        }

    return data_dict

In [10]:
train_set_dict = add_artificial_mask(
        train_set_X, artificial_missing_rate, "train"
    )
val_set_dict = add_artificial_mask(val_set_X, artificial_missing_rate, "val")
test_set_dict = add_artificial_mask(
        test_set_X, artificial_missing_rate, "test"
    )

processed_data = {
        "train": train_set_dict,
        "val": val_set_dict,
        "test": test_set_dict,
    }

print(f"Feature num: {df.columns},")
print(f'Sample num in train set: {len(train_set_dict["X"])}')
print(f'Sample num in val set: {len(val_set_dict["X"])}')
print(f'Sample num in test set: {len(test_set_dict["X"])}')

Feature num: Index(['BS', 'CALI', 'DENS', 'DRHO', 'DTC', 'GR', 'NEUT', 'PEF', 'RESD',
       'RESM', 'RESS', 'SP', 'TEMP', 'TENS', 'DEPT', 'WELL'],
      dtype='object'),
Sample num in train set: 38837
Sample num in val set: 13036
Sample num in test set: 12396


In [11]:
print("验证集缺失率：", np.isnan(val_set_dict['X_hat']).sum() / val_set_dict['X_hat'].size)
print("测试集缺失率：", np.isnan(test_set_dict['X_hat']).sum() / test_set_dict['X_hat'].size)

验证集缺失率： 0.5209585251099519
测试集缺失率： 0.5285072604065828


In [12]:

def saving_into_h5(saving_dir, data_dict, classification_dataset):
    """Save data into h5 file.
    Parameters
    ----------
    saving_dir: path of saving dir
    data_dict: data dictionary containing train/val/test sets
    classification_dataset: boolean, if this is a classification dataset
    """

    def save_each_set(handle, name, data):
        single_set = handle.create_group(name)
        if classification_dataset:
            single_set.create_dataset("labels", data=data["labels"].astype(int))
        single_set.create_dataset("X", data=data["X"].astype(np.float32))
        if name in ["val", "test"]:
            single_set.create_dataset("X_hat", data=data["X_hat"].astype(np.float32))
            single_set.create_dataset(
                "missing_mask", data=data["missing_mask"].astype(np.float32)
            )
            single_set.create_dataset(
                "indicating_mask", data=data["indicating_mask"].astype(np.float32)
            )

    saving_path = os.path.join(saving_dir, "datasets.h5")
    with h5py.File(saving_path, "w") as hf:
        hf.create_dataset(
            "empirical_mean_for_GRUD",
            data=data_dict["train"]["empirical_mean_for_GRUD"],
        )
        save_each_set(hf, "train", data_dict["train"])
        save_each_set(hf, "val", data_dict["val"])
        save_each_set(hf, "test", data_dict["test"])


In [13]:
def pickle_dump(data, path):
    """ Pickle the given object.

    Parameters
    ----------
    data : object
        The object to be pickled.

    path : string,
        Saving path.

    Returns
    -------
    `path` if succeed else None

    """
    try:
        with open(path, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    except pickle.PicklingError:
        print('Pickling failed. No cache will be saved.')
        return None
    return path

In [15]:
saving_into_h5(dataset_saving_dir, processed_data, classification_dataset=False)
pickle_dump(scaler, os.path.join(dataset_saving_dir, 'scaler'))

'../generated_datasets/NewZealand_seqlen100_02masked/scaler'